# Przetworzenie obrazów przez GroundingSAM

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

In [5]:
# Dodanie dźwięku przy długich komórkach
import subprocess
def play_sound():
    sound_file = '/mnt/d/Backup/INZ/msg.ogg'
    subprocess.run(['ffplay', '-nodisp', '-autoexit', sound_file], capture_output=True)

In [6]:
sys.path.append(str(Path().resolve() / "Grounded-SAM-2"))

from torchvision.ops import box_convert
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from grounding_dino.groundingdino.util.inference import load_model, load_image, predict
import torch

2026-03-05 19:26:59.539847: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-05 19:27:01.242187: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Ustawienie opcji wyświetlania w pandas (opcjonalne)

In [7]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [8]:
SAVE_DIR = Path("/mnt/d/Backup/MAGISTERSKIE/outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

## Pobranie danych

Pobranie ścieżek do plików w odpowiedniej kolejności. Następnie połączenie z danymi w pliku .csv tak, aby sobie odpowidały.

In [12]:
DATA_PATH = Path("/mnt/d/Backup/INZ/data")

# Załoadowanie CSV
df = pd.read_csv(DATA_PATH / "2011_data.csv", sep=";")

# Folder z obrazami
image_folder = Path(DATA_PATH / "2011_img")

# Funkcja zwracająca ścieżkę do obrazu kolorowego
def get_color_path_1(row):
    return image_folder / f"rlm_rosbag_2024_12_06-10_27_18_{int(row['id'])}_color_orig.png"

# Funkcja zwracająca ścieżkę do obrazu głębi
def get_depth_path_1(row):
    return image_folder / f"rlm_rosbag_2024_12_06-10_27_18_{int(row['id'])}_depth.png"

# Dodanie kolumn z ścieżkami
df['color_path'] = df.apply(get_color_path_1, axis=1)
df['depth_path'] = df.apply(get_depth_path_1, axis=1)

# Wyświetlenie pierwszych 12 wierszy
df.head()

,id,img_base,ESJoint1,ESJoint2,ESJoint3,ESJoint4,ESJoint5,ESJoint6,gripper_finger_1_joint,gripper_finger_2_joint,color_path,depth_path
0,0,rlm_rosbag_2024_12_06-10_27_18_1,3.847648,0.730016,1.842955,1.703969,0.990853,3.103483,0.792556,0.72097,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_0_color_orig.png,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_0_depth.png
1,1,rlm_rosbag_2024_12_06-10_27_18_1,3.847648,0.730016,1.842955,1.703969,0.990853,3.103483,0.792556,0.72097,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_1_color_orig.png,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_1_depth.png
2,2,rlm_rosbag_2024_12_06-10_27_18_2,3.847648,0.730192,1.842994,1.703825,0.990661,3.103483,0.792556,0.72097,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2_color_orig.png,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2_depth.png
3,3,rlm_rosbag_2024_12_06-10_27_18_3,3.847648,0.730329,1.843014,1.703393,0.990565,3.103339,0.792556,0.72097,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_3_color_orig.png,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_3_depth.png
4,4,rlm_rosbag_2024_12_06-10_27_18_4,3.847648,0.730388,1.843034,1.703010,0.990326,3.103291,0.792556,0.72097,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_4_color_orig.png,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_4_depth.png


In [13]:
df.tail()

,id,img_base,ESJoint1,ESJoint2,ESJoint3,ESJoint4,ESJoint5,ESJoint6,gripper_finger_1_joint,gripper_finger_2_joint,color_path,depth_path
2007,2007,rlm_rosbag_2024_12_06-10_27_18_2007,3.786284,2.415651,1.128585,0.978108,0.646331,2.022554,0.306796,1.206730,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2007_color_orig.png,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2007_depth.png
2008,2008,rlm_rosbag_2024_12_06-10_27_18_2008,3.836703,2.253075,1.313933,0.962241,0.784437,1.966228,0.255663,1.278315,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2008_color_orig.png,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2008_depth.png
2009,2009,rlm_rosbag_2024_12_06-10_27_18_2009,3.977662,2.088345,1.543924,1.079495,0.716031,2.035305,0.731196,0.761876,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2009_color_orig.png,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2009_depth.png
2010,2010,rlm_rosbag_2024_12_06-10_27_18_2010,4.187857,2.094435,1.691756,0.987839,0.584396,2.080749,0.787442,0.726083,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2010_color_orig.png,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2010_depth.png
2011,2011,rlm_rosbag_2024_12_06-10_27_18_2011,4.189385,1.908519,1.835143,1.038988,0.552806,2.071641,0.547119,0.966406,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2011_color_orig.png,/mnt/d/Backup/INZ/data/2011_img/rlm_rosbag_2024_12_06-10_27_18_2011_depth.png


Zapisanie dataframe do pliku .csv

In [14]:
#df.to_csv(str(SAVE_DIR)+'/df.csv', index=False)

# Konfiguracja GDSAM

In [8]:
df = pd.read_csv(str(SAVE_DIR)+'/df.csv')
df.shape

(2012, 12)

In [6]:
# Wczytanie do stałych ścieżek do modeli i konfiguracji
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"
SAM2_CHECKPOINT = "Grounded-SAM-2/checkpoints/sam2.1_hiera_large.pt"
GROUNDING_DINO_CONFIG = "Grounded-SAM-2/grounding_dino/groundingdino/config/GroundingDINO_SwinB_cfg.py"
GROUNDING_DINO_CHECKPOINT = "Grounded-SAM-2/gdino_checkpoints/groundingdino_swinb_cogcoor.pth"

# Ustawienie odpowiednich parametrów
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.get_device_properties(0).major >= 8:
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [7]:
# załadaowanie modelu SAM
sam2_model = build_sam2(SAM2_CONFIG, SAM2_CHECKPOINT, device=DEVICE)
sam2_predictor = SAM2ImagePredictor(sam2_model)

# załadaowanie modelu GD
grounding_model = load_model(
    model_config_path=GROUNDING_DINO_CONFIG, 
    model_checkpoint_path=GROUNDING_DINO_CHECKPOINT,
    device=DEVICE
)

final text_encoder_type: bert-base-uncased


# Detekcja i segmentacja

In [12]:
incomplete_detect = []
masks_array = []
all_boxes = np.zeros((len(df),3,4), dtype=float)  # Znormalizowane współrzędne [0,1]

# Tekstowe prompty dla detekcji
TEXT_PROMPTS = ["pink robot finger gripper on the robot gripper.", "green small rectangle on the robot gripper.", "blue robot finger gripper on the robot gripper."]

# Próg detekcji dla boxów i tekstu
BOX_TRESHOLD = 0.28
TEXT_TRESHOLD = 0.3

# Parametr odfiltrowania dużych masek
AREA_THRESHOLD = 2300

for count_iter, i in enumerate(range(0, len(df))):

  # Załadowanie obrazu kolorowego
  image_source, image = load_image(df['color_path'][i])
  h, w, _ = image_source.shape
  sam2_predictor.set_image(image_source)

  # Załadowanie obrazu głębokości
  depth_img, _ = load_image(df['depth_path'][i])
  depth_img = depth_img[:, :, 0]
  
  # Tymczasowa tablica do przechowywania masek
  tmp = np.zeros((len(TEXT_PROMPTS), h, w), dtype=np.float32)

  # Iteracja po trzech obiektach
  for j in range(len(TEXT_PROMPTS)):

    # Wybór tekstu dla obiektu
    text_prompt = TEXT_PROMPTS[j]

    # Detekcja obiektów
    boxes, confidences, labels = predict(
        model=grounding_model,
        image=image,
        caption=text_prompt,
        box_threshold=BOX_TRESHOLD,
        text_threshold=TEXT_TRESHOLD)

    # Konwersja boxów z cxcywh na xyxy
    boxes2 = boxes * torch.Tensor([w, h, w, h])
    xyxy = box_convert(boxes=boxes2, in_fmt="cxcywh", out_fmt="xyxy").numpy()

    # Sprawdzenie czy są detekcje
    print(f"Obraz {i}, Obiekt {j}: Liczba wykrytych boxów: {len(xyxy)}")
    print(f"xyxy = {xyxy}")
    if xyxy.size == 0:
        print(f"Brak detekcji dla '{text_prompt}' na obrazie {i}")
        incomplete_detect.append(i)
        continue

    # Filtrowanie boxów po rozmiarze (przed segmentacją)
    if j == 1: max_wh = 80
    else: max_wh = 60
    
    valid_box_indices = []
    for idx in range(len(xyxy)):
        box_width = xyxy[idx][2] - xyxy[idx][0]
        box_height = xyxy[idx][3] - xyxy[idx][1]
        if box_width <= max_wh and box_height <= max_wh:
            valid_box_indices.append(idx)
    
    # Sprawdzenie czy pozostały jakieś boxy po filtrowaniu
    if len(valid_box_indices) == 0:
        print(f"Obraz {i}, Obiekt {j}: Wszystkie boxy przekraczają limit {max_wh}px")
        incomplete_detect.append(i)
        continue
    
    # Wybór tylko przefiltrowanych boxów do segmentacji
    valid_box_indices = np.array(valid_box_indices)
    filtered_xyxy = xyxy[valid_box_indices]
    filtered_confidences = confidences[valid_box_indices]
    filtered_labels = [labels[idx] for idx in valid_box_indices]
    
    print(f"Po filtrowaniu pozostało {len(filtered_xyxy)} boxów")

    # Segmentacja obiektów (tylko na przefiltrowanych boxach)
    masks, scores, logits = sam2_predictor.predict(
        point_coords=None,
        point_labels=None,
        box=filtered_xyxy,
        multimask_output=False,
    )

    """
    Pzetwarzanie po detekcji i segmentacji
    """

    # Zamiana wymiarów do (n, H, W)
    if masks.ndim == 4:
        masks = masks.squeeze(1)

    # Obliczenie pola segmentacji
    mask_areas = np.sum(masks, axis=(1, 2))  # Obliczenie pola segmentacji
    valid_mask_indices = np.where(mask_areas < AREA_THRESHOLD)[0] # Zachowanie segmentacji poniżej 2300 pikseli

    # Sprawdzenie czy są maski
    if len(valid_mask_indices) == 0:
        print(f"Obraz {i}, Obiekt {j}: Brak maski z polem < {AREA_THRESHOLD} pikseli")
        incomplete_detect.append(i)
        continue 

    # Wybór maski o największym współczynniku pewności (z pozostałych kandydatów)
    filtered_confidences_np = filtered_confidences.numpy()
    valid_confidences = filtered_confidences_np[valid_mask_indices]
    best_mask_idx = valid_mask_indices[np.argmax(valid_confidences)]

    # Wybór najlepszej maski i boxa
    selected_mask = masks[best_mask_idx:best_mask_idx+1]  # Zachowanie wymiarów (1, H, W)
    selected_xyxy = filtered_xyxy[best_mask_idx:best_mask_idx+1]  # Zachowanie wymiarów (1, 4)
    selected_confidence = filtered_confidences_np[best_mask_idx]
    selected_label = filtered_labels[best_mask_idx]
    selected_area = mask_areas[best_mask_idx]
    
    box_width = selected_xyxy[0][2] - selected_xyxy[0][0]
    box_height = selected_xyxy[0][3] - selected_xyxy[0][1]
    
    # Zapisanie boxa odpowiadającego wybranej masce (znormalizowane wartości)
    normalized_box = selected_xyxy[0].copy()
    normalized_box[0] /= w  # x_min
    normalized_box[1] /= h  # y_min
    normalized_box[2] /= w  # x_max
    normalized_box[3] /= h  # y_max
    all_boxes[i][j] = normalized_box

    # Wyświetlenie informacji
    print(f"Numer obrazu: {i}")
    print(f"Tekst wejściowy: {text_prompt}")
    print(f"Maska - Pole: {selected_area} pikseli, Pewność: {selected_confidence:.2f}")
    print(f"Box - Szerokość: {box_width:.1f}px, Wysokość: {box_height:.1f}px")
    print(f"Box znormalizowany: {normalized_box}")

    # Zapisanie maski do tablicy
    mask = selected_mask[0]
    tmp[j] = mask

    """
    Wizualizacja i wyświetlanie
    """

    # # Zmienne do wizualizacji
    # class_ids = np.array([0]) 
    # labels = [f"{selected_label} {selected_confidence:.2f}"]

    # # Detekcje do wizuazlizacji
    # detections = sv.Detections(
    #     xyxy=selected_xyxy,  # (1, 4)
    #     mask=selected_mask.astype(bool),  # (1, H, W)
    #     class_id=class_ids
    # )

    # # Wizualizacja prostokątów
    # box_annotator = sv.BoxAnnotator()
    # annotated_frame = box_annotator.annotate(scene=cv2.cvtColor(image_source, cv2.COLOR_BGR2RGB), detections=detections)

    # # Wizualizacja etykiet
    # label_annotator = sv.LabelAnnotator()
    # annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

    # # Wizualizacja masek
    # mask_annotator = sv.MaskAnnotator()
    # annotated_frame = mask_annotator.annotate(scene=annotated_frame, detections=detections)
    # sv.plot_image(annotated_frame)

  # Zapisanie masek do tablicy
  masks_array.append(tmp)

masks_array = np.array(masks_array)
print(f"masks_array -> {masks_array.shape}")

print(f"Lista niekompletnych id: {incomplete_detect}")

# Niekompletne detekcjeprint(f"Liczba niekompletnych id: {incomplete_detect.size}. Liczba iteracji: {count_iter+1}. Procent odrzuconych: {incomplete_detect.size/(count_iter+1)*100}%")

incomplete_detect = np.array(list(set(incomplete_detect)))
incomplete_detect = np.unique(incomplete_detect)

print(f"Liczba niekompletnych id: {incomplete_detect.size}. Liczba iteracji: {count_iter+1}. Procent odrzuconych: {incomplete_detect.size/(count_iter+1)*100}%")
print(f"Lista niekompletnych id: {incomplete_detect}")

sound_file = '/mnt/d/Backup/INZ/msg.ogg'
_ = subprocess.run(['ffplay', '-nodisp', '-autoexit', sound_file], capture_output=True)

Obraz 0, Obiekt 0: Liczba wykrytych boxów: 1
xyxy = [[143.68495   88.045044 168.27602  134.71042 ]]
Po filtrowaniu pozostało 1 boxów
Numer obrazu: 0
Tekst wejściowy: pink robot finger gripper on the robot gripper.
Maska - Pole: 305.0 pikseli, Pewność: 0.30
Box - Szerokość: 24.6px, Wysokość: 46.7px
Box znormalizowany: [0.28063467 0.2076534  0.3286641  0.31771326]
Obraz 0, Obiekt 1: Liczba wykrytych boxów: 1
xyxy = [[  1.2712555 135.99341   510.16693   380.38794  ]]
Obraz 0, Obiekt 1: Wszystkie boxy przekraczają limit 80px
Obraz 0, Obiekt 2: Liczba wykrytych boxów: 2
xyxy = [[121.115364 112.26334  160.91948  141.51332 ]
 [142.20758  120.74701  159.45676  139.89893 ]]
Po filtrowaniu pozostało 2 boxów
Numer obrazu: 0
Tekst wejściowy: blue robot finger gripper on the robot gripper.
Maska - Pole: 771.0 pikseli, Pewność: 0.32
Box - Szerokość: 39.8px, Wysokość: 29.2px
Box znormalizowany: [0.23655345 0.26477203 0.31429586 0.33375782]
Obraz 1, Obiekt 0: Liczba wykrytych boxów: 1
xyxy = [[143.684

Zapisanie tablic z danymi do pliku

In [17]:
# np.save(SAVE_DIR / 'masks_array.npy', masks_array)
# np.save(SAVE_DIR / 'incomplete_detect.npy', incomplete_detect)
# np.save(SAVE_DIR / 'all_boxes.npy', all_boxes)

In [ ]:
masks_array.shape

# Wizualizacja wyników

In [15]:
masks_array = np.load(SAVE_DIR / 'masks_array.npy')
incomplete_detect = np.load(SAVE_DIR / 'incomplete_detect.npy')
all_boxes = np.load(SAVE_DIR / 'all_boxes.npy')

play_sound()

In [47]:
type(all_boxes.shape[0])

int

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import random #random.randint(0,len(incomplete_detect)-1)

test_idx = random.randint(0,all_boxes.shape[0]-1)
test_row = df.iloc[test_idx]

# Wczytanie obrazu kolorowego
color_image = cv2.imread(str(test_row['color_path']))
color_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
h, w = color_image.shape[:2]

# Tworzenie obrazu z nałożonymi maskami
plt.figure(figsize=(15, 10))
plt.imshow(color_image)

# Nałożenie masek z przezroczystością - z wygenerowanych danych
red_mask = masks_array[test_idx][0]  # Pink gripper
green_mask = masks_array[test_idx][1]  # Green rectangle
blue_mask = masks_array[test_idx][2]  # Blue gripper

plt.imshow(red_mask, cmap='Reds', alpha=0.35)
plt.imshow(green_mask, cmap='Greens', alpha=0.35)
plt.imshow(blue_mask, cmap='Blues', alpha=0.35)

# Rysowanie bounding boxów z wygenerowanych danych
colors = ['red', 'green', 'blue']
labels_text = ['Czerwony', 'Zielony', 'Niebieski']

for i, color in enumerate(colors):
    x_min_norm, y_min_norm, x_max_norm, y_max_norm = all_boxes[test_idx][i]
    
    # Sprawdzenie czy box jest niepusty
    if x_min_norm == 0 and y_min_norm == 0 and x_max_norm == 0 and y_max_norm == 0:
        print(f"Box {i} jest pusty")
        continue
    
    # Konwersja ze znormalizowanych do pikseli
    x_min_pixel = int(x_min_norm * w)
    y_min_pixel = int(y_min_norm * h)
    x_max_pixel = int(x_max_norm * w)
    y_max_pixel = int(y_max_norm * h)
    
    rect = patches.Rectangle((x_min_pixel, y_min_pixel),
                            x_max_pixel - x_min_pixel,
                            y_max_pixel - y_min_pixel,
                            linewidth=2, edgecolor=color, facecolor='none',
                            label=f"{labels_text[i]} element")
    plt.gca().add_patch(rect)

plt.title(f"Obraz o ID: {test_row['id']} - Maski i otaczające prostokąty")
plt.legend(loc='upper right')
plt.axis('off')
plt.tight_layout()
plt.show()